# 04 — Session Patterns

Heatmap: hour × weekday from events table. Timing recommendations for reminders.

**Outputs:** `analysis/outputs/session_heatmap.png`

In [ ]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == "analysis" else os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import asyncio
from analysis.lib.pa_charts import plot_session_heatmap
from db import get_db, init_db
from services import AnalyticsService

OUTPUT_DIR = os.path.join(REPO_ROOT, "analysis", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
async def load_heatmap(days=30):
    db_path = os.path.join(REPO_ROOT, "studybuddy.db")
    if not os.path.exists(db_path):
        return {}
    db = await get_db(db_path)
    await init_db(db)
    analytics = AnalyticsService(db)
    data = await analytics.compute_heatmap(days=days)
    await db.close()
    return data

heatmap = asyncio.run(load_heatmap())
print("Peak:", heatmap.get("peak"))
print("Total events:", heatmap.get("total_events"))

In [ ]:
grid = heatmap.get("grid", [])
if grid and heatmap.get("total_events", 0) > 0:
    fig = plot_session_heatmap(
        grid,
        heatmap.get("weekday_labels", []),
        heatmap.get("hour_labels", []),
        title="Palph Activity Heatmap (events)",
    )
    out = os.path.join(OUTPUT_DIR, "session_heatmap.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved: {out}")
else:
    print("No heatmap data yet.")

## Key findings

- Peak hour: ___
- Peak day: ___
- Reminder timing recommendation: ___